# Karnak-40B test for the Political Predictor

Runs the program's own three AI steps (AI trait analysis, the 16 stakeholder reactions, and a direct
political question) on the public Trump UNGA 2025 speech, in Arabic and English, with **Karnak-40B**
(Applied Innovation Center, Apache 2.0) on Kaggle's GPUs.

**Before running:** in the right-hand panel set **Accelerator: GPU T4 x2** and turn **Internet on**
(needs a phone-verified Kaggle account). Then click **Run all**. It takes roughly 30 to 60 minutes,
most of it the 23 GB model download.

**At the end:** download `karnak40_results.zip` from the Output panel and send it back.

In [ ]:
# 1. Check the GPUs and free disk space.
!nvidia-smi --query-gpu=name,memory.total --format=csv
!df -h / /kaggle/working | tail -n 2

In [ ]:
# 2. Install Ollama (runs AI models on this machine) and start it on both GPUs.
!apt-get -qq update && apt-get -qq install -y zstd pciutils > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh
import os, subprocess, time, urllib.request
env = dict(os.environ,
           OLLAMA_HOST="127.0.0.1:11434",
           OLLAMA_CONTEXT_LENGTH="16384",   # room for the program's longest prompt
           OLLAMA_NUM_PARALLEL="1",
           OLLAMA_KEEP_ALIVE="-1",          # keep the model loaded between steps
           OLLAMA_FLASH_ATTENTION="1",
           OLLAMA_KV_CACHE_TYPE="q8_0",     # smaller memory for the reading cache
           OLLAMA_SCHED_SPREAD="1")         # spread the model over both GPUs
server = subprocess.Popen(["ollama", "serve"], env=env, stdout=open("ollama.log", "w"), stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        urllib.request.urlopen("http://127.0.0.1:11434/api/version", timeout=2); break
    except Exception:
        time.sleep(2)
print(urllib.request.urlopen("http://127.0.0.1:11434/api/version").read().decode())

In [ ]:
# 3. Download Karnak-40B (compressed i1-Q4_K_S, 23.2 GB) and check it answers.
MODEL = None
for tag in ("i1-Q4_K_S", "Q4_K_S"):
    name = "hf.co/mradermacher/Karnak-i1-GGUF:" + tag
    if subprocess.run(["ollama", "pull", name]).returncode == 0:
        MODEL = name
        break
assert MODEL, "The model download failed; see the messages above."
print("model:", MODEL)
!ollama run {MODEL} "قل مرحبا في كلمة واحدة"
!ollama ps

In [ ]:
# 4. Unpack the program's AI code, the two speeches and the test script.
import base64, io, zipfile
BUNDLE = "..."
zipfile.ZipFile(io.BytesIO(base64.b64decode(BUNDLE))).extractall("/kaggle/working/pp")
!pip -q install httpx
!ls -R /kaggle/working/pp | head -20

In [ ]:
# 5. Run the test: same prompts and settings the program uses with Groq.
import subprocess, os
run_env = dict(os.environ,
               LLM_BASE_URL="http://127.0.0.1:11434/v1", LLM_MODEL=MODEL,
               LLM_TIMEOUT="1800", LLM_MAX_INPUT_TOKENS="7500",
               PYTHONIOENCODING="utf-8", PYTHONPATH="/kaggle/working/pp")
os.makedirs("/kaggle/working/results", exist_ok=True)
for lang in ("ar", "en"):
    subprocess.run(["python", "/kaggle/working/pp/karnak_test.py", "/kaggle/working/results",
                    f"karnak40_{lang}", f"/kaggle/working/pp/speeches/trump_{lang}.txt"],
                   cwd="/kaggle/working/pp", env=run_env)

In [ ]:
# 6. Pack the results for download, with a short summary.
import json, glob, shutil
for f in sorted(glob.glob("/kaggle/working/results/*.json")):
    d = json.load(open(f, encoding="utf-8"))
    print(f.split("/")[-1], {k: (d[k]["ok"], d[k]["seconds"]) for k in ("traits", "reactions", "politics")})
!nvidia-smi --query-gpu=name,memory.used --format=csv
shutil.copy("ollama.log", "/kaggle/working/results/ollama.log")
shutil.make_archive("/kaggle/working/karnak40_results", "zip", "/kaggle/working/results")
print("Download karnak40_results.zip from the Output panel.")